# Feature Investigation — Churn Drivers Analysis

## Objective
Identify key factors that influence customer churn by analyzing relationships between features and churn behavior.

In [2]:
# Load cleaned dataset for analysis
import pandas as pd

df = pd.read_csv("E:/Customer Churn Analysis/data/processed/telco_churn_cleaned.csv")

c:\Users\mahdi\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\mahdi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
df.groupby("paymentmethod")["churn_flag"].mean().sort_values(ascending=False)

paymentmethod
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: churn_flag, dtype: float64

In [4]:
df.groupby("internetservice")["churn_flag"].mean().sort_values(ascending=False)

internetservice
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: churn_flag, dtype: float64

In [5]:
# Rank categorical features by churn rate
categorical_features = [
    'contract', 'paymentmethod', 'internetservice',
    'techsupport', 'onlinesecurity', 'gender', 'seniorcitizen',
    'partner', 'dependents', 'multiplelines'
]

feature_churn_rates = {}
for feature in categorical_features:
    churn_rate = df.groupby(feature)['churn_flag'].mean().sort_values(ascending=False)
    feature_churn_rates[feature] = churn_rate

# Display churn rates for each feature
for feature, rates in feature_churn_rates.items():
    print(f'Churn rate by {feature}:')
    print(rates)

Churn rate by contract:
contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: churn_flag, dtype: float64
Churn rate by paymentmethod:
paymentmethod
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: churn_flag, dtype: float64
Churn rate by internetservice:
internetservice
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: churn_flag, dtype: float64
Churn rate by techsupport:
techsupport
No     0.311862
Yes    0.151663
Name: churn_flag, dtype: float64
Churn rate by onlinesecurity:
onlinesecurity
No     0.313296
Yes    0.146112
Name: churn_flag, dtype: float64
Churn rate by gender:
gender
Female    0.269209
Male      0.261603
Name: churn_flag, dtype: float64
Churn rate by seniorcitizen:
seniorcitizen
1    0.416813
0    0.236062
Name: churn_flag, dtype: float64
Churn rate by partner:
partner
No     0.329580
Yes    0

Customer churn is primarily driven by contract flexibility, low engagement services, and payment behavior rather than demographic factors.

In [5]:
cols = ["techsupport", "onlinesecurity"]

for col in cols:
    print(df.groupby(col)["churn_flag"].mean())

techsupport
No     0.311862
Yes    0.151663
Name: churn_flag, dtype: float64
onlinesecurity
No     0.313296
Yes    0.146112
Name: churn_flag, dtype: float64


In [ ]:
# Analyse churn rate across monthly charges quartiles
df['monthlycharge_bin'] = pd.qcut(df['monthlycharges'], q=4, duplicates='drop')
monthly_churn = df.groupby('monthlycharge_bin')['churn_flag'].mean().sort_values(ascending=False)
print('Churn rate by monthly charges (quartiles):')
print(monthly_churn)

Churn rate by monthly charges (quartiles):
monthlycharge_bin
(70.35, 89.85]     0.375071
(89.85, 118.75]    0.328783
(35.5, 70.35]      0.245753
(18.249, 35.5]     0.112372
Name: churn_flag, dtype: float64


Customers with higher monthly charges exhibit significantly higher churn rates, indicating strong price sensitivity and potential value mismatch.

In [8]:
# Construct a simple rule-based risk score
df['risk_score'] = 0
df.loc[df['contract'] == 'Month-to-month', 'risk_score'] += 2
df.loc[df['tenure'] <= 12, 'risk_score'] += 2
df.loc[df['paymentmethod'] == 'Electronic check', 'risk_score'] += 1
df.loc[df['techsupport'] == 'No', 'risk_score'] += 1
df.loc[df['onlinesecurity'] == 'No', 'risk_score'] += 1
df.loc[df['monthlycharges'] > df['monthlycharges'].median(), 'risk_score'] += 1

# Assign risk levels
def assign_risk_level(score):
    if score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'

df['risk_level'] = df['risk_score'].apply(assign_risk_level)

# Summarize churn rate and revenue by risk level
risk_summary = df.groupby('risk_level').agg(
    customer_count=('customerid', 'count'),
    churn_rate=('churn_flag', 'mean'),
    average_monthly_charge=('monthlycharges', 'mean'),
    revenue_at_risk=('total_revenue', 'sum')
).sort_values(by='churn_rate', ascending=False)
print(risk_summary)

            customer_count  churn_rate  average_monthly_charge  \
risk_level                                                       
High                  2239    0.537740               64.729768   
Medium                2254    0.240905               69.927684   
Low                   2550    0.047843               60.223392   

            revenue_at_risk  
risk_level                   
High             1763217.75  
Medium           5792229.80  
Low              8499643.90  


The high-risk segment has the highest churn rate (~53%) and represents a significant revenue exposure. However, the medium-risk segment contains the largest potential for churn reduction impact due to its high revenue base.

# 📌 Summary — Feature Investigation

## 🎯 Key Findings

The analysis identified four main drivers of customer churn:

- **Contract Type:** Month-to-month customers show the highest churn rate.
- **Tenure:** Customers in the early stage of their lifecycle are more likely to churn.
- **Payment Method:** Electronic check users have significantly higher churn rates.
- **Service Usage:** Lack of Tech Support and Online Security increases churn probability.

---

## 🧠 Business Interpretation

Churn is primarily driven by customer behavior and engagement rather than demographic factors. Customers with low commitment, limited service adoption, and weak engagement are more likely to leave.

---

## 🎯 Business Action

- Focus on early customer retention strategies
- Encourage long-term contracts
- Promote automated payment methods
- Increase adoption of support and security services

---

## 🚀 Final Insight

Churn is concentrated in low-engagement customers, especially those with flexible contracts and limited service usage.